# 공연 활동과 외식업 매출: 업종 차이와 GEE
## 목적과 해석 범위
김태경 교수 공저 논문의 **업종 간 관계와 수혜 유형별 차이**라는 분석틀을 공연과 외식업에 적용한다. 원 논문 모형을 그대로 재현하는 작업은 아니다.

- 범위: 종로구·마포구, 한식음식점·커피음료, 2023~2025년 12분기.
- 기본 분석: 500m 관측 등록 공연 수와 총매출·21~24시 매출 비중의 관련성. 공연 수×업종 상호작용을 직접 검정한다.
- 기본 평균모형: 상권×업종 고정효과 + 업종×연도분기 고정효과. 표준오차는 상권 군집으로 계산한다.
- 300/800m, 동일 상권·분기에서 두 업종이 관측되는 표본, 두 업종 12분기 균형표본, 점포 수 통제는 민감도 분석이다.
- GEE는 Gaussian identity / 상권 군집 / exchangeable 구조를 사용하며 상권 내 변화와 상권 간 차이를 분리한다. 이 세부 설정은 원 논문의 명시된 설정이 아니라 본 프로젝트의 선택이다.
- 식별되지 않은 인과효과, 실제 상연 횟수·관객 수, 일별 소비로 해석하지 않는다.

**실행 전:** 01~03번 노트북 산출물이 필요하다. 이 노트북은 인터넷/API를 호출하지 않는다. 원자료를 변경하지 않는다.

**기존 두 업종 기준선:** 이 노트북은 `dataset/analysis/baseline_two_industry/`에 고정한 재수집 전 입력으로 실행한다. 최신 외식업 10종 분석은 08~10번을 사용한다.


In [ ]:
# 현재 구조 및 과거 고정 명세의 경로를 함께 해석합니다.
from pathlib import Path as _ProjectPath
import json as _bootstrap_json
_project_home = next(p for p in [_ProjectPath.cwd(), *_ProjectPath.cwd().parents]
                     if (p / 'environment/project_paths.ipynb').is_file())
_project_notebook = _bootstrap_json.loads((_project_home / 'environment/project_paths.ipynb').read_text(encoding='utf-8'))
for _project_cell in _project_notebook['cells']:
    if 'project-runtime' in _project_cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_project_cell['source']), 'environment/project_paths.ipynb', 'exec'), globals())

from pathlib import Path
import os
# 큰 더미 행렬의 계산에 과도한 스레드를 만들지 않는다.
for name in ['OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'OMP_NUM_THREADS']:
    os.environ[name] = '1'
import json, hashlib, warnings, platform
import importlib.metadata as metadata
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

ROOT = PROJECT_ROOT
REPORTS, DATA = project_path('reports/20261005'), project_path('dataset/analysis')
FIGURES = REPORTS / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
INPUT = DATA / 'baseline_two_industry/performance_sales_panel.csv'
input_sha256 = hashlib.sha256(INPUT.read_bytes()).hexdigest()
panel = pd.read_csv(INPUT, dtype={'area_id': str, 'industry_id': str, 'quarter': str})
KEY = ['area_id', 'industry_id', 'quarter']
assert not panel.duplicated(KEY).any()
assert set(panel['industry_id']) == {'CS100001', 'CS100010'}
assert set(panel['quarter']) == {f'{y}{q}' for y in range(2023, 2026) for q in range(1, 5)}
pd.set_option('display.max_columns', 14)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
print(f'입력 {len(panel):,}행 / 원본 해시 {input_sha256[:12]}…')


## 1. 변수 정의와 표본
공연 활동은 시설 위치가 확보된 **서로 다른 등록 공연 ID 수**다. 매출 미제공은 결측으로 유지한다. 야간 비중의 분모가 0이거나 값이 범위를 벗어나면 그 결과변수 분석에서 제외한다.

점포 수는 공연과 매출 사이의 매개 경로일 수 있으므로 기본모형에서 제외한다. 기존 03번 분석과 비교하는 별도 모형에서만 점포 수와 점포 수×업종을 통제한다. 모든 금액은 명목 원화다.


In [ ]:
panel['entity'] = panel['area_id'] + '_' + panel['industry_id']
panel['cafe'] = panel['industry_id'].eq('CS100010').astype(int)
panel['log_sales'] = np.log1p(panel['total_sales'].where(panel['total_sales'].ge(0)))
panel['night_share_pct'] = (100 * panel['night_sales'] / panel['total_sales']).where(
    panel['total_sales'].gt(0) & panel['night_sales'].ge(0)
    & panel['night_sales'].le(panel['total_sales']))
panel['log_stores'] = np.log1p(panel['store_count'].where(panel['store_count'].ge(0)))
panel['stores_cafe'] = panel['log_stores'] * panel['cafe']
panel['mapo_cafe'] = panel['district'].eq('마포구').astype(int) * panel['cafe']
quarter_terms = []
for quarter in sorted(panel['quarter'].unique())[1:]:
    name = f'q_{quarter}'
    panel[name] = panel['quarter'].eq(quarter).astype(int)
    panel[f'{name}_cafe'] = panel[name] * panel['cafe']
    quarter_terms += [name, f'{name}_cafe']
for radius in [300, 500, 800]:
    col = f'known_performance_count_{radius}m'
    assert panel[col].notna().all() and panel[col].ge(0).all()
assert (panel['known_performance_count_300m'] <= panel['known_performance_count_500m']).all()
assert (panel['known_performance_count_500m'] <= panel['known_performance_count_800m']).all()
assert panel.loc[panel['total_sales'].isna(), 'log_sales'].isna().all()
panel.to_csv(DATA / 'industry_interaction_panel.csv', index=False, encoding='utf-8-sig')
display(panel.groupby('industry', observed=True).agg(
    전체행수=('quarter', 'size'), 매출관측=('total_sales', 'count'),
    야간비중관측=('night_share_pct', 'count'), 상권수=('area_id', 'nunique')))


## 2. 분석 계획
500m 전체 관측표본에서 두 결과변수의 **카페 - 한식 기울기 차이**를 기본 검정으로 둔다. 이 두 검정에는 Holm 보정 p값도 함께 제시한다. 거리·표본·점포 수·GEE 결과는 탐색적 민감도 분석이며 유의한 결과만 선택하지 않는다.

각 모형의 표본 수·상권 수·제외 수를 기록한다. 한 번만 관측된 상권×업종은 시간 내 변화 분석에서 제외한다. 기본표본에서는 업종별 매출 제공 구성이 다를 수 있어 공통 관측표본과 균형표본을 별도로 확인한다.


In [ ]:
SCENARIOS = [
    ('main_500', 500, 'all', False, '기본 500m'),
    ('radius_300', 300, 'all', False, '거리 300m'),
    ('radius_800', 800, 'all', False, '거리 800m'),
    ('paired_500', 500, 'paired', False, '두 업종 동시 관측'),
    ('balanced_500', 500, 'balanced', False, '두 업종 12분기 균형'),
    ('stores_500', 500, 'all', True, '점포 수 통제'),
]
OUTCOMES = {'log_sales': '총매출 log(1+원)', 'night_share_pct': '21~24시 매출 비중(%p)'}
sample_rows, sample_keys, result_rows, diagnostics = [], [], [], []
models, frames = {}, {}

def make_frame(outcome, radius, sample_kind, stores=False):
    frame = panel.copy()
    frame['log_activity'] = np.log1p(frame[f'known_performance_count_{radius}m'])
    frame['activity_cafe'] = frame['log_activity'] * frame['cafe']
    required = [outcome, 'log_activity', *KEY]
    if stores:
        required += ['log_stores']
    frame = frame.dropna(subset=required)
    if sample_kind in ['paired', 'balanced']:
        n = frame.groupby(['area_id', 'quarter'])['industry_id'].transform('nunique')
        frame = frame.loc[n.eq(2)]
    if sample_kind == 'balanced':
        n = frame.groupby('area_id')['quarter'].transform('nunique')
        frame = frame.loc[n.eq(12)]
    n = frame.groupby('entity')['quarter'].transform('nunique')
    return frame.loc[n.ge(2)].sort_values(KEY).copy()


## 3. 고정효과 모형과 기울기 차이
업종 주효과는 상권×업종 고정효과에 이미 흡수된다. 중복 더미를 피하기 위해 11개 분기 더미와 각각의 카페 상호작용을 명시적으로 생성했다.

한식 기울기는 β, 카페 기울기는 β+δ, 업종 차이는 δ다. 카페 신뢰구간은 β와 δ의 공분산을 포함한 선형대조로 계산한다. 군집 보정 표준오차와 상권 군집 수−1 자유도의 t 추론을 사용한다.


In [ ]:
def fe_formula(outcome, stores=False):
    terms = ['log_activity', 'activity_cafe', 'C(entity)', *quarter_terms]
    if stores:
        terms += ['log_stores', 'stores_cafe']
    return outcome + ' ~ ' + ' + '.join(terms)

def extract_contrasts(model, scenario, outcome, radius, frame,
                      main_term='log_activity', interaction='activity_cafe', estimator='FE'):
    terms = list(model.params.index)
    specs = {'한식음식점': {main_term: 1},
             '커피음료': {main_term: 1, interaction: 1},
             '카페 - 한식 차이': {interaction: 1}}
    rows = []
    for label, weights in specs.items():
        vector = np.array([weights.get(term, 0) for term in terms], dtype=float)
        test = model.t_test(vector)
        ci = np.asarray(test.conf_int()).ravel()
        rows.append(dict(scenario=scenario, outcome=outcome, estimator=estimator,
                         radius_m=radius, contrast=label, coefficient=float(np.asarray(test.effect).item()),
                         standard_error=float(np.asarray(test.sd).item()),
                         ci_low=float(ci[0]), ci_high=float(ci[1]),
                         p_value=float(np.asarray(test.pvalue).item()), nobs=len(frame),
                         area_clusters=frame['area_id'].nunique()))
    return rows


In [ ]:
for scenario, radius, sample_kind, stores, label in SCENARIOS:
    for outcome in OUTCOMES:
        frame = make_frame(outcome, radius, sample_kind, stores)
        assert frame['area_id'].nunique() >= 20
        model = smf.ols(fe_formula(outcome, stores), data=frame).fit(
            cov_type='cluster', cov_kwds={'groups': frame['area_id'],
                                         'use_correction': True, 'df_correction': True},
            use_t=True)
        rank = np.linalg.matrix_rank(model.model.exog)
        assert rank == model.model.exog.shape[1], f'{scenario}: 중복 설계열'
        models[(scenario, outcome)] = model
        frames[(scenario, outcome)] = frame
        result_rows += extract_contrasts(model, scenario, outcome, radius, frame)
        sample_rows.append(dict(scenario=scenario, label=label, outcome=outcome, nobs=len(frame),
                                area_clusters=frame['area_id'].nunique(),
                                entities=frame['entity'].nunique(), excluded_rows=len(panel)-len(frame),
                                stores_controlled=stores, radius_m=radius))
        sample_keys.append(frame[KEY].assign(scenario=scenario, outcome=outcome))
        diagnostics.append(dict(scenario=scenario, outcome=outcome, estimator='FE',
                                design_columns=model.model.exog.shape[1], rank=rank,
                                residual_df=float(model.df_resid),
                                inference_df=float(model.df_resid_inference)))
display(pd.DataFrame(sample_rows).query("outcome == 'log_sales'"))


## 4. 기본 결과와 업종 차이
결과표에서 '카페 - 한식 차이'의 신뢰구간과 p값을 직접 확인한다. 신뢰구간에 0이 포함되는 것은 두 업종의 관계가 같다는 증명이 아니라, 현재 자료에서 차이를 뚜렷하게 확인하지 못했다는 뜻이다.


In [ ]:
results = pd.DataFrame(result_rows)
results['p_holm_primary'] = np.nan
primary_mask = results['scenario'].eq('main_500') & results['contrast'].eq('카페 - 한식 차이')
results.loc[primary_mask, 'p_holm_primary'] = multipletests(
    results.loc[primary_mask, 'p_value'], method='holm')[1]
display(results.loc[results['scenario'].eq('main_500'),
                    ['outcome', 'contrast', 'coefficient', 'ci_low', 'ci_high',
                     'p_value', 'p_holm_primary', 'nobs', 'area_clusters']])


## 5. 구현 검증
통합 상호작용 모형은 기울기를 업종별로 자유롭게 허용하므로, 같은 표본에서 업종별로 따로 추정한 기울기와 일치해야 한다. 공분산까지 동일하다는 의미는 아니다. 이 검사는 상호작용·분기 더미 구현 오류를 찾는 데 사용한다.


In [ ]:
validation = []
for scenario in ['main_500', 'stores_500']:
    for outcome in OUTCOMES:
        pooled = models[(scenario, outcome)]
        frame = frames[(scenario, outcome)]
        for cafe, industry in [(0, '한식음식점'), (1, '커피음료')]:
            sub = frame.loc[frame['cafe'].eq(cafe)]
            formula = f'{outcome} ~ log_activity + C(area_id) + C(quarter)'
            if scenario == 'stores_500':
                formula += ' + log_stores'
            separate = smf.ols(formula, data=sub).fit()
            target = pooled.params['log_activity'] + cafe * pooled.params['activity_cafe']
            error = abs(target - separate.params['log_activity'])
            assert error < 1e-7, (scenario, outcome, industry, error)
            validation.append(dict(check='pooled_equals_separate_slope',
                                   scenario=scenario, outcome=outcome, industry=industry,
                                   absolute_difference=float(error), passed=True))
display(pd.DataFrame(validation))


## 6. GEE 보조 분석
상권 고정효과를 포함한 Gaussian exchangeable GEE는 OLS와 점추정이 구조적으로 같아질 수 있다. 또한 군집 내 잔차합 제약 때문에 상관계수가 경계로 추정될 수 있다. 따라서 여기서는 공연 활동을 **상권×업종별 관측기간 평균(상권 간 수준)**과 **그 평균에서 벗어난 값(상권 내 변화)**으로 나눈다.

GEE에는 두 변수와 업종별 기울기, 업종, 지역×업종, 업종×분기 효과를 넣는다. 상권×업종 고정효과는 넣지 않는다. FE와 평균모형의 통제 방식이 다르므로 결과 차이가 추정기 자체만의 차이라는 뜻은 아니다. 이 within-between 설정 역시 관측되지 않은 모든 교란을 제거하지 않는다.

동일 상권 안의 두 업종과 시계열을 하나의 군집으로 묶는다. Gaussian identity / exchangeable / robust sandwich, 정규근사 95% 구간을 사용한다. 수렴 여부·작업상관계수 α·최대 군집의 양의 정부호 조건을 점검하며, 실패한 추정치는 결과표에 넣지 않는다.

같은 상권의 업종별 공연 평균이 다를 수 있는 이유는 매출이 제공된 분기의 차이 때문이다. Exchangeable은 상권 안의 모든 관측쌍에 같은 작업상관을 가정한다. 강건 표준오차는 이 상관 가정 오류를 완화하지만 누락변수·선택적 결측·상권 간 공간 상관을 해결하지 않는다.


In [ ]:
gee_rows, gee_diagnostics = [], []
for outcome in OUTCOMES:
    frame = frames[('main_500', outcome)].copy()
    frame['between_activity'] = frame.groupby('entity')['log_activity'].transform('mean')
    frame['within_activity'] = frame['log_activity'] - frame['between_activity']
    frame['within_cafe'] = frame['within_activity'] * frame['cafe']
    frame['between_cafe'] = frame['between_activity'] * frame['cafe']
    terms = ['within_activity', 'within_cafe', 'between_activity', 'between_cafe',
             'cafe', 'C(district)', 'mapo_cafe', *quarter_terms]
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        gee = smf.gee(outcome + ' ~ ' + ' + '.join(terms),
                      groups=frame['area_id'], data=frame,
                      family=sm.families.Gaussian(),
                      cov_struct=sm.cov_struct.Exchangeable()).fit(maxiter=100, cov_type='robust')
    alpha = float(gee.cov_struct.dep_params)
    max_cluster = int(frame.groupby('area_id').size().max())
    min_eigenvalue = min(1-alpha, 1+(max_cluster-1)*alpha)
    rank = np.linalg.matrix_rank(gee.model.exog)
    valid = bool(gee.converged and min_eigenvalue > 1e-8
                 and rank == gee.model.exog.shape[1]
                 and np.isfinite(gee.params).all() and np.isfinite(gee.bse).all())
    gee_diagnostics.append(dict(outcome=outcome, converged=bool(gee.converged),
        alpha=alpha, max_cluster_size=max_cluster, min_working_eigenvalue=min_eigenvalue,
        rank=rank, design_columns=gee.model.exog.shape[1], accepted=valid,
        warnings=' | '.join(str(w.message) for w in caught)))
    if valid:
        gee_rows += extract_contrasts(gee, 'gee_within_between_500', outcome, 500, frame,
                                     'within_activity', 'within_cafe', 'GEE')
        models[('gee_within_between_500', outcome)] = gee
display(pd.DataFrame(gee_diagnostics))
if gee_rows:
    display(pd.DataFrame(gee_rows)[['outcome', 'contrast', 'coefficient', 'ci_low', 'ci_high', 'p_value']])


## 7. 기울기와 신뢰구간 시각화
아래는 log(1+관측 공연 수)의 계수다. 로그 공연 수 1단위는 공연 1개를 의미하지 않는다. 야간 비중 계수는 퍼센트포인트이며, 음의 계수가 야간 매출액 감소를 뜻하지는 않는다.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
font_names = {f.name for f in fm.fontManager.ttflist}
plt.rcParams['font.family'] = next((f for f in ['Malgun Gothic', 'NanumGothic', 'AppleGothic']
                                   if f in font_names), 'DejaVu Sans')
plt.rcParams.update({'axes.unicode_minus': False, 'figure.dpi': 110, 'font.size': 10})
all_results = pd.concat([results, pd.DataFrame(gee_rows)], ignore_index=True)
COLORS = {'한식음식점': '#27628B', '커피음료': '#C56C37', '카페 - 한식 차이': '#65539A'}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
for ax, outcome in zip(axes, OUTCOMES):
    part = results.loc[results['scenario'].eq('main_500') & results['outcome'].eq(outcome)]
    for i, row in enumerate(part.itertuples()):
        ax.errorbar(row.coefficient, i, xerr=[[row.coefficient-row.ci_low], [row.ci_high-row.coefficient]],
                    fmt='o', capsize=4, color=COLORS[row.contrast])
    ax.set_yticks(range(len(part)), part['contrast'])
    ax.axvline(0, color='#777777', linestyle='--', linewidth=1)
    ax.set_title(OUTCOMES[outcome]); ax.set_xlabel('기울기 및 95% 신뢰구간')
    ax.grid(axis='x', alpha=.2); ax.invert_yaxis()
fig.suptitle('500m 기본모형: 공연 활동과 매출의 관련성')
fig.tight_layout()
fig.savefig(FIGURES / '05_기본모형_업종차이.png', dpi=160, bbox_inches='tight')
plt.show(); plt.close(fig)


## 8. 거리·표본·점포 수·평균모형을 바꿨을 때
각 줄은 카페와 한식의 기울기 차이를 나타낸다. GEE는 평균모형과 추론 방식까지 달라 별도 보조 결과로 구분한다. 거리별 노출은 같은 공연장을 여러 상권에 연결할 수 있어 상권 간 공간 상관을 제거하지 못한다.


In [ ]:
labels = {s[0]: s[4] for s in SCENARIOS}
labels['gee_within_between_500'] = 'GEE: 상권 내·간 분리'
fig, axes = plt.subplots(1, 2, figsize=(12, 5.4))
for ax, outcome in zip(axes, OUTCOMES):
    part = all_results.loc[all_results['outcome'].eq(outcome)
                           & all_results['contrast'].eq('카페 - 한식 차이')]
    for i, row in enumerate(part.itertuples()):
        ax.errorbar(row.coefficient, i,
                    xerr=[[row.coefficient-row.ci_low], [row.ci_high-row.coefficient]],
                    fmt='s' if row.estimator == 'GEE' else 'o', capsize=3,
                    color='#C56C37' if row.estimator == 'GEE' else '#27628B')
    ax.set_yticks(range(len(part)), [labels[s] for s in part['scenario']])
    ax.axvline(0, color='#777777', linestyle='--', linewidth=1)
    ax.set_title(OUTCOMES[outcome]); ax.set_xlabel('카페 - 한식 기울기 차이 및 95% 구간')
    ax.invert_yaxis(); ax.grid(axis='x', alpha=.2)
fig.suptitle('설정에 따른 업종 차이: 탐색적 민감도 분석')
fig.tight_layout()
fig.savefig(FIGURES / '05_민감도_업종차이.png', dpi=160, bbox_inches='tight')
plt.show(); plt.close(fig)


## 9. 공연 수 10→20개 비교 시나리오
계수를 단순 탄력성으로 부르지 않는다. 공연 10개에서 20개로 변할 때 log(1+공연 수)의 차이는 log(21/11)이다. 이는 설정한 비교 시나리오이지 실제 개입 효과가 아니다. 비교 구간이 각 업종 관측자료 범위 안에 있는지도 확인한다.

역변환 값은 조건부 (총매출+1)의 기하평균 상대차이다. 기대 총매출의 산술평균 변화율로 해석하려면 잔차 재변환에 추가 가정이 필요하다. 비중 결과는 퍼센트포인트 변화다.


In [ ]:
change_x = np.log(21/11)
contrast_changes = []
for row in results.loc[results['scenario'].eq('main_500')
                       & results['contrast'].isin(['한식음식점', '커피음료'])].itertuples():
    interval = np.array([row.coefficient, row.ci_low, row.ci_high]) * change_x
    if row.outcome == 'log_sales':
        interval = 100 * np.expm1(interval)
        unit = '(총매출+1) 기하평균 상대차이 %'
    else:
        unit = '야간 비중 변화 %p'
    contrast_changes.append(dict(outcome=row.outcome, industry=row.contrast, comparison='공연 10→20개',
                                 estimated_change=interval[0], ci_low=interval[1], ci_high=interval[2], unit=unit))
change_table = pd.DataFrame(contrast_changes)
display(change_table)
support = frames[('main_500', 'log_sales')].groupby('industry', observed=True)['known_performance_count_500m'].agg(['min','median','max'])
display(support)


## 10. 결과 요약·재현 기록
매출 결측, 시설명 임시 연결, 좌표 미수집, 현재 경계의 과거 적용을 보완하기 전까지 예비 관련성 분석이다. 두 업종의 차이를 찾지 못해도 연구 실패를 뜻하지 않는다. 효과가 작거나 자료의 측정·표본 한계 때문에 불확실할 수 있다.

원 논문 출처: Mun, Woo, Kim & Shin (2026), *Can nearby high-quality restaurants increase other tourism firms' performance? Evidence from hotels*, IJHM 132, 104356. DOI: 10.1016/j.ijhm.2025.104356. 방법 pp.6–10. 원 논문의 2SLS는 유효한 도구변수가 확보되지 않아 구현하지 않았다.


In [ ]:
all_results.to_csv(REPORTS / '05_interaction_results.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(sample_rows).to_csv(REPORTS / '05_model_samples.csv', index=False, encoding='utf-8-sig')
pd.concat(sample_keys).to_csv(REPORTS / '05_sample_membership.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(diagnostics).to_csv(REPORTS / '05_fe_diagnostics.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(gee_diagnostics).to_csv(REPORTS / '05_gee_diagnostics.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(validation).to_csv(REPORTS / '05_implementation_checks.csv', index=False, encoding='utf-8-sig')
change_table.to_csv(REPORTS / '05_activity_change_contrasts.csv', index=False, encoding='utf-8-sig')
summary_lines = []
for row in results.loc[primary_mask].itertuples():
    finding = '차이를 관측했다' if row.p_holm_primary < .05 else '차이를 뚜렷하게 확인하지 못했다'
    summary_lines.append(f"- **{OUTCOMES[row.outcome]}**: 카페 - 한식 기울기 차이 {row.coefficient:.4f}, "
                         f"95% 구간 [{row.ci_low:.4f}, {row.ci_high:.4f}], "
                         f"p={row.p_value:.4f}, Holm p={row.p_holm_primary:.4f}. 현재 자료에서 {finding}.")
display(Markdown('\n'.join(summary_lines)))
metadata_record = dict(input_file=str(INPUT.relative_to(ROOT)), input_sha256=input_sha256,
    scope='종로구·마포구 / 한식·커피음료 / 2023~2025 분기',
    primary_radius_m=500, fe_models=len(SCENARIOS)*len(OUTCOMES), accepted_gee_models=len(gee_rows)//3,
    python=platform.python_version(), packages={name: metadata.version(name)
        for name in ['pandas','numpy','statsmodels','scipy','matplotlib']},
    causal_identification=False, gee_is_exact_paper_replication=False,
    limitations=['시설명 잠정 연결','시설 좌표 미수집','현재 상권 경계의 과거 적용',
                 '등록 공연 수는 상연 횟수·관객 수와 다름','상권 간 공간 상관 미해결',
                 '매출 미제공에 따른 표본 선택','점포 수 통제는 잠재적 매개경로 조건화'])
metadata_record['output_sha256'] = {
    str((REPORTS / name).relative_to(ROOT)).replace('\\', '/'):
        hashlib.sha256((REPORTS / name).read_bytes()).hexdigest()
    for name in ['05_interaction_results.csv', '05_model_samples.csv',
                 '05_fe_diagnostics.csv', '05_gee_diagnostics.csv',
                 '05_implementation_checks.csv']}
(REPORTS / '05_analysis_summary.json').write_text(
    json.dumps(metadata_record, ensure_ascii=False, indent=2), encoding='utf-8')
assert hashlib.sha256(INPUT.read_bytes()).hexdigest() == input_sha256


## 11. 개별 업종 결과도 표본 변경과 함께 해석
한식의 야간 비중 계수가 특정 모형에서 유의해도, 카페와의 차이까지 유의하다는 뜻은 아니다. 아래 표는 같은 결과를 공통 관측표본·균형표본·거리 변경으로 확인한다. 비중 하락을 야간 매출액 감소나 공연의 부정적 인과효과로 해석하지 않는다.

In [ ]:
night_sensitivity = results.loc[
    results['outcome'].eq('night_share_pct') & results['contrast'].eq('한식음식점'),
    ['scenario', 'coefficient', 'ci_low', 'ci_high', 'p_value', 'nobs']].copy()
night_sensitivity['설정'] = night_sensitivity['scenario'].map(labels)
display(night_sensitivity[['설정', 'coefficient', 'ci_low', 'ci_high', 'p_value', 'nobs']])